# 06 - Benchmark: FPS, latency, memory, power, precision / recall / F1, and Llama speed  (run on the Jetson)


In [7]:
import sys, json, time, importlib, platform, subprocess, urllib.request
from pathlib import Path
REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO))
import cv2, numpy as np
from edge_inspect import settings, severity as sv, detector as dt, llama_advisor as la, pipeline as pl

In [8]:
%%writefile ../edge_inspect/benchmark_utils.py
"""benchmark_utils.py - latency stats, RAM sampling, Jetson power via tegrastats. Python 3.8+."""
from __future__ import annotations

import re
import shutil
import statistics
import subprocess
import threading
import time
from typing import Dict, List, Optional

try:
    import psutil
except ImportError:
    psutil = None


def summarize(ms: List[float]) -> dict:
    if not ms:
        return {}
    s = sorted(ms)
    q = lambda f: s[min(len(s) - 1, int(round(f * (len(s) - 1))))]
    return {"n": len(s), "mean": round(statistics.mean(s), 3), "p50": round(q(0.5), 3),
            "p95": round(q(0.95), 3), "min": round(s[0], 3), "max": round(s[-1], 3)}


class TegraMonitor:
    """Background `tegrastats` sampler: RAM used (MB) and board power VDD_IN (mW). No-op off Jetson."""
    RAM = re.compile(r"RAM (\d+)/(\d+)MB")
    PWR = re.compile(r"VDD_IN (\d+)mW(?:/(\d+)mW)?")

    def __init__(self, interval_ms: int = 500):
        self.interval_ms, self.proc, self.samples = interval_ms, None, []
        self.available = shutil.which("tegrastats") is not None

    def start(self):
        if not self.available:
            return self
        self.proc = subprocess.Popen(["tegrastats", "--interval", str(self.interval_ms)],
                                     stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
        threading.Thread(target=self._read, daemon=True).start()
        return self

    def _read(self):
        for line in self.proc.stdout:
            ram, pwr = self.RAM.search(line), self.PWR.search(line)
            self.samples.append({"ram_mb": int(ram.group(1)) if ram else None,
                                 "power_mw": int(pwr.group(1)) if pwr else None})

    def stop(self) -> dict:
        if self.proc:
            self.proc.terminate()
        rams = [s["ram_mb"] for s in self.samples if s["ram_mb"] is not None]
        pw = [s["power_mw"] for s in self.samples if s["power_mw"] is not None]
        return {"tegrastats": self.available, "samples": len(self.samples),
                "ram_used_mb_peak": max(rams) if rams else None,
                "ram_used_mb_mean": round(statistics.mean(rams), 1) if rams else None,
                "power_mw_mean": round(statistics.mean(pw), 1) if pw else None,
                "power_mw_peak": max(pw) if pw else None}


def process_rss_mb() -> Optional[float]:
    return round(psutil.Process().memory_info().rss / 1e6, 1) if psutil else None

Overwriting ../edge_inspect/benchmark_utils.py


In [9]:
from edge_inspect import benchmark_utils as bu
for m in (sv, dt, la, pl, bu): importlib.reload(m)
S = settings.load(REPO)
meta = json.loads(Path(S["meta"]).read_text()); cfg = sv.load_config(S["severity_config"])

NUM_WARMUP, NUM_RUNS, N_LLAMA = 10, 200, 8
det = dt.OnnxYoloDetector(S["onnx"], meta["classes"], imgsz=meta["imgsz"], conf=meta["deploy_conf"], iou=meta["nms_iou"])
files = sorted(p for p in Path(S["demo_images"]).glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
imgs = [cv2.imread(str(f)) for f in files]
backend = la.RunnerBackend(S["llama_main"], S["llama_model"], S["llama_tokenizer"], temperature=0.2)
print(len(imgs), "images | providers:", det.providers, "| Llama runner available:", backend.available())

def sh(c):
    try: return subprocess.run(c, shell=True, capture_output=True, text=True, timeout=10).stdout.strip()
    except Exception: return ""
device = {"model": sh("cat /proc/device-tree/model | tr -d '\\0'"), "jetpack_release": sh("head -1 /etc/nv_tegra_release"),
          "power_mode": sh("nvpmodel -q 2>/dev/null | tr '\\n' ' '") or "unknown (run: sudo nvpmodel -q)",
          "python": platform.python_version(), "onnx_providers": det.providers}
print(device)

29 images | providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'] | Llama runner available: True
{'model': 'NVIDIA Jetson Orin Nano Engineering Reference Developer Kit Super', 'jetpack_release': '# R36 (release), REVISION: 5.2, GCID: 46426093, BOARD: generic, EABI: aarch64, DATE: Thu Jul 16 18:56:22 UTC 2026', 'power_mode': 'NV Power Mode: 25W 1', 'python': '3.10.12', 'onnx_providers': ['CUDAExecutionProvider', 'CPUExecutionProvider']}


## 1) Vision: per-stage latency, FPS, RAM and power


In [10]:
det.warmup(NUM_WARMUP)
idle = bu.TegraMonitor(500).start(); time.sleep(8); idle_stats = idle.stop()
mon = bu.TegraMonitor(500).start()
stage = {"pre": [], "infer": [], "post": [], "severity": [], "vision_total": []}
for i in range(NUM_RUNS):
    img = imgs[i % len(imgs)]
    dets, tm = det.infer(img)
    t0 = time.perf_counter()
    sv.build_handoff("bench.jpg", dets, img.shape[1], img.shape[0], cfg, conf_min=det.conf)
    sev_ms = (time.perf_counter() - t0) * 1e3
    stage["pre"].append(tm["pre_ms"]); stage["infer"].append(tm["infer_ms"]); stage["post"].append(tm["post_ms"])
    stage["severity"].append(sev_ms); stage["vision_total"].append(tm["pre_ms"] + tm["infer_ms"] + tm["post_ms"] + sev_ms)
vis_hw = mon.stop()
vis_mean = float(np.mean(stage["vision_total"]))
vision = {"images_timed": NUM_RUNS, "fps_mean": round(1000.0 / vis_mean, 2),
          "fps_p95_worst_case": round(1000.0 / float(np.percentile(stage["vision_total"], 95)), 2),
          "stage_ms": {k: bu.summarize(v) for k, v in stage.items()},
          "python_process_rss_mb": bu.process_rss_mb(), "hardware_during": vis_hw, "hardware_idle": idle_stats}
print(f"Vision FPS (mean) {vision['fps_mean']} | slowest 5% of frames {vision['fps_p95_worst_case']} FPS")
for k, v in vision["stage_ms"].items(): print(f"{k:13s} mean {v['mean']:8.2f} ms   p95 {v['p95']:8.2f} ms")
print("hardware:", vis_hw, "| idle:", idle_stats)

Vision FPS (mean) 37.12 | slowest 5% of frames 23.76 FPS
pre           mean     5.28 ms   p95    10.04 ms
infer         mean    20.09 ms   p95    34.29 ms
post          mean     1.18 ms   p95     2.04 ms
severity      mean     0.39 ms   p95     0.72 ms
vision_total  mean    26.94 ms   p95    42.06 ms
hardware: {'tegrastats': True, 'samples': 10, 'ram_used_mb_peak': 4627, 'ram_used_mb_mean': 4622.9, 'power_mw_mean': 8057.4, 'power_mw_peak': 8584} | idle: {'tegrastats': True, 'samples': 15, 'ram_used_mb_peak': 4633, 'ram_used_mb_mean': 4625.6, 'power_mw_mean': 5210.3, 'power_mw_peak': 5910}


## 2) Llama: time to first token, tokens/s, total response time, RAM


In [11]:
llama, pipe_e2e = None, None
if backend.available():

    handoffs = []
    for img, f in zip(imgs, files):
        dets, _ = det.infer(img)
        h = sv.build_handoff(f.name, dets, img.shape[1], img.shape[0], cfg, conf_min=det.conf)
        if h["needs_advice"]: handoffs.append(h)
        if len(handoffs) >= N_LLAMA: break
    mon = bu.TegraMonitor(500).start()
    rows, e2e = [], []
    for h in handoffs:
        t0 = time.perf_counter()
        a = la.advise(h, backend)
        wall = time.perf_counter() - t0
        L = a.get("llm") or {}
        rows.append({"source": a["source"], "ttft_s": L.get("ttft_s") or L.get("first_output_s"), "decode_tps": L.get("decode_tps"),
                     "prefill_tps": L.get("prefill_tps"), "total_s": L.get("total_s"), "load_s": L.get("model_load_s"),
                     "prompt_tokens": L.get("prompt_tokens"), "generated_tokens": L.get("generated_tokens"), "peak_rss_mb": L.get("peak_rss_mb")})
        e2e.append(wall)
        print(f"{h['image_id']:22s} {a['source']:18s} total {wall:6.1f}s  ttft {rows[-1]['ttft_s']}  tok/s {rows[-1]['decode_tps']}")
    llama_hw = mon.stop()
    col = lambda k: [r[k] for r in rows if r[k] is not None]
    llama = {"calls": len(rows), "fallback_calls": sum(r["source"] != "llama" for r in rows),
             "ttft_s": bu.summarize(col("ttft_s")), "decode_tokens_per_s": bu.summarize(col("decode_tps")),
             "prefill_tokens_per_s": bu.summarize(col("prefill_tps")), "total_response_s": bu.summarize(col("total_s")),
             "model_load_s": bu.summarize(col("load_s")), "runner_peak_rss_mb": max(col("peak_rss_mb")) if col("peak_rss_mb") else None,
             "hardware_during": llama_hw,
             "settings": {"model": "Llama-3.2-1B-Instruct 8da4w (ExecuTorch, XNNPACK CPU)", "temperature": 0.2, "max_new_tokens": 110}}
    pipe_e2e = {"image_to_advice_s": bu.summarize([ (vis_mean / 1000.0) + w for w in e2e ])}
    print(json.dumps(llama, indent=2))
else:
    print("Llama runner not available - finish notebook 02, then re-run this cell. (TTFT / tokens/s stay empty until then.)")

crazing_104.jpg        template_fallback  total   20.8s  ttft 5.887  tok/s 11.869149860079128
crazing_12.jpg         template_fallback  total   35.4s  ttft 5.16  tok/s 11.728395061728394
inclusion_110.jpg      llama              total   16.6s  ttft 5.504  tok/s 13.201320132013201
inclusion_115.jpg      llama              total   18.6s  ttft 5.692  tok/s 12.521069106669879
inclusion_12.jpg       llama              total   16.7s  ttft 4.874  tok/s 13.314556394827807
inclusion_122.jpg      template_fallback  total   19.6s  ttft 5.684  tok/s 13.277499744663466
inclusion_123.jpg      llama              total   23.9s  ttft 5.44  tok/s 12.31390009739808
patches_1.jpg          llama              total   17.4s  ttft 5.758  tok/s 12.415797120591732
{
  "calls": 8,
  "fallback_calls": 3,
  "ttft_s": {
    "n": 8,
    "mean": 5.5,
    "p50": 5.684,
    "p95": 5.887,
    "min": 4.874,
    "max": 5.887
  },
  "decode_tokens_per_s": {
    "n": 8,
    "mean": 12.58,
    "p50": 12.521,
    "p95": 13.31

## 3) Is the speed enough for the use case? (fill in your own assumptions)


In [12]:
LINE_SPEED_M_S = 1.0
CROP_LENGTH_M = 0.20
required = LINE_SPEED_M_S / CROP_LENGTH_M
verdict = "meets" if vision["fps_p95_worst_case"] >= required else "does NOT meet"
print(f"Required: {required:.1f} FPS | measured mean {vision['fps_mean']} FPS, slowest 5% {vision['fps_p95_worst_case']} FPS -> {verdict} the requirement")
use_case = {"line_speed_m_s": LINE_SPEED_M_S, "crop_length_m": CROP_LENGTH_M, "required_fps": round(required, 2), "verdict": verdict}

Required: 5.0 FPS | measured mean 37.12 FPS, slowest 5% 23.76 FPS -> meets the requirement


## 4) Detector quality (from notebook 00) and the final report


In [13]:
report = {
    "created": time.strftime("%Y-%m-%dT%H:%M:%S"), "device": device,
    "detector_quality_validation": {**meta["val_metrics"], "note": meta["split"], "classwise": meta["classwise_val"]},
    "vision": vision, "llama": llama, "pipeline_end_to_end": pipe_e2e, "use_case_justification": use_case,
}
out = Path(S["runs_dir"]); out.mkdir(exist_ok=True)
(out / "benchmark_report.json").write_text(json.dumps(report, indent=2))

q = report["detector_quality_validation"]
md_lines = ["| Metric | Value |", "|---|---|",
            f"| Precision / Recall / F1 (macro, conf {q['conf']}) | {q['precision_macro']} / {q['recall_macro']} / {q['f1_macro']} |",
            f"| mAP50 / mAP50-95 | {q['mAP50']} / {q['mAP50_95']} |",
            f"| Vision FPS (mean / slowest 5%) | {vision['fps_mean']} / {vision['fps_p95_worst_case']} |",
            f"| Per-image vision latency ms (mean / p95) | {vision['stage_ms']['vision_total']['mean']} / {vision['stage_ms']['vision_total']['p95']} |",
            f"| Power while running, mW (mean) vs idle | {vis_hw.get('power_mw_mean')} vs {idle_stats.get('power_mw_mean')} |",
            f"| RAM used, MB (peak, whole board) | {vis_hw.get('ram_used_mb_peak')} |"]
if llama:
    md_lines += [f"| Llama time to first token, s (mean / p95) | {llama['ttft_s'].get('mean')} / {llama['ttft_s'].get('p95')} |",
                 f"| Llama decode tokens/s (mean) | {llama['decode_tokens_per_s'].get('mean')} |",
                 f"| Llama total response time, s (mean / p95) | {llama['total_response_s'].get('mean')} / {llama['total_response_s'].get('p95')} |",
                 f"| Llama runner peak RAM, MB | {llama['runner_peak_rss_mb']} |"]
(out / "benchmark_report.md").write_text("\n".join(md_lines))
print("\n".join(md_lines))

try:
    req = urllib.request.Request(S["tracker_url"].rstrip("/") + "/api/benchmark", data=json.dumps(report).encode(), headers={"Content-Type": "application/json"})
    urllib.request.urlopen(req, timeout=2).read(); print("\nreport posted to the tracker")
except Exception:
    print("\ntracker not running; report saved to", out / "benchmark_report.json")

| Metric | Value |
|---|---|
| Precision / Recall / F1 (macro, conf 0.268) | 0.7377 / 0.7634 / 0.7351 |
| mAP50 / mAP50-95 | 0.8088 / 0.5087 |
| Vision FPS (mean / slowest 5%) | 37.12 / 23.76 |
| Per-image vision latency ms (mean / p95) | 26.938 / 42.059 |
| Power while running, mW (mean) vs idle | 8057.4 vs 5210.3 |
| RAM used, MB (peak, whole board) | 4627 |
| Llama time to first token, s (mean / p95) | 5.5 / 5.887 |
| Llama decode tokens/s (mean) | 12.58 |
| Llama total response time, s (mean / p95) | 21.123 / 35.372 |
| Llama runner peak RAM, MB | 1172.3 |

report posted to the tracker
